# 🚨 CrisisCompute — Multi-Agent Negotiation Training

> **OpenEnv Hackathon 2026 · Theme #1 (Multi-Agent Interactions) + Theme #4 (Self-Improvement)**

This notebook trains three RL agents — `data_loader`, `data_cleaner`, `ml_trainer` — inside a **compute-allocation negotiation environment**. Agents compete for constrained GPU / CPU resources across 8-hour episodes, negotiate contracts, form coalitions, and handle crisis injections (GPU outages, urgent-task spikes).

**What you'll see:**
- All 5 curriculum levels, every one with negotiation enabled
- Reward curve showing improvement over 50 episodes
- Holdout evaluation: trained vs untrained agent comparison
- Full action-distribution + Q-table analysis
- HuggingFace TRL-compatible training loop

---
**Curriculum Levels:**

| Level | Scenario | Negotiation | Crisis |
|-------|----------|-------------|--------|
| 0 | Stable market | ✅ | ❌ |
| 1 | Light GPU outage | ✅ | ✅ |
| 2 | Urgent task injection | ✅ | ✅ |
| 3 | Early outage + urgency | ✅ | ✅ |
| 4 | Compound crisis | ✅ | ✅ |

## 1. Setup & Install Dependencies

In [ ]:
# Check if we're in Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone the repo (replace with your actual HuggingFace Space URL when deployed)
    !git clone https://github.com/YOUR_USERNAME/CrisisCompute.git
    %cd CrisisCompute
    print("✅ Repo cloned")
else:
    import os
    # If running locally, adjust path as needed
    project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
    os.chdir(project_root)
    print(f"✅ Working directory: {os.getcwd()}")

In [ ]:
%%capture install_output
# Install all required packages
!pip install -q \
    numpy>=1.21.0 \
    matplotlib>=3.5.0 \
    requests>=2.28.0 \
    python-dotenv>=0.19.0 \
    scipy>=1.7.0 \
    pandas>=1.3.0 \
    openenv-core>=0.2.0 \
    fastapi>=0.110.0 \
    uvicorn>=0.29.0 \
    pydantic>=2.0.0 \
    gradio>=4.0.0 \
    groq>=0.4.0

print("✅ Dependencies installed")

## 2. Configure API Keys & Environment

In [ ]:
import os

# ── Option A: Use Colab Secrets (recommended) ──────────────────────────────
try:
    from google.colab import userdata
    GROQ_KEY = userdata.get('GROQ_API_KEY')
    if GROQ_KEY:
        os.environ['GROQ_API_KEY'] = GROQ_KEY
        print("✅ GROQ_API_KEY loaded from Colab Secrets")
except Exception:
    pass

# ── Option B: Paste directly (NOT recommended for shared notebooks) ─────────
# os.environ['GROQ_API_KEY'] = 'gsk_...'     # ← paste your key here if needed

# ── Training Configuration ──────────────────────────────────────────────────
# All 5 levels always have negotiation ON — this is the core of Theme #1
os.environ['NUM_EPISODES']             = '50'      # 50 episodes per run
os.environ['TRAINING_AGENT_MODE']      = 'rl'      # 'rl' | 'llm' | 'hybrid'
os.environ['LLM_PROVIDER']             = 'groq'    # groq | ollama | huggingface | openrouter
os.environ['GROQ_MODEL']               = 'llama-3.1-8b-instant'
os.environ['NEGOTIATION_ENABLED']      = 'true'    # Always on for all levels
os.environ['CRISIS_MODE_ENABLED']      = 'false'   # Overridden per-level by curriculum
os.environ['SELF_IMPROVEMENT_ENABLED'] = 'true'    # Theme #4 adaptive curriculum
os.environ['SHOW_PHASE_LOGS']          = 'true'    # Show per-phase promotion status
os.environ['RL_WARMUP_EPISODES']       = '25'      # Q-table warmup episodes
os.environ['MULTI_SEED']               = '42'      # Fixed seed for reproducibility
os.environ['BASELINE_EPISODES_CAP']    = '15'      # Cap baseline comparison run
os.environ['SEED_EPISODES_CAP']        = '15'      # Cap per-seed eval

print("")
print("═" * 60)
print("  CrisisCompute · Training Configuration")
print("═" * 60)
print(f"  Episodes            : {os.environ['NUM_EPISODES']}")
print(f"  Agent Mode          : {os.environ['TRAINING_AGENT_MODE'].upper()}")
print(f"  LLM Provider        : {os.environ['LLM_PROVIDER'].upper()}")
print(f"  Negotiation         : ALL 5 levels ✅")
print(f"  Adaptive Curriculum : {os.environ['SELF_IMPROVEMENT_ENABLED']}")
print(f"  Seed                : {os.environ['MULTI_SEED']}")
print("═" * 60)

## 3. Environment Verification

Confirm the RL-Friendly Environment loads correctly and all 5 curriculum levels are wired with negotiation.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

# ── 1. Load environment ────────────────────────────────────────────────────
try:
    from satya_env.rl_environment import RLFriendlyEnvironment
    env = RLFriendlyEnvironment(config_dir=None, seed=42)
    obs = env.reset()
    print("✅ RLFriendlyEnvironment loaded")
    print(f"   Agents   : {list(obs.keys()) if isinstance(obs, dict) else 'non-dict obs'}")
    print(f"   Negotiation enabled: {getattr(env, 'negotiation_enabled', 'N/A')}")
except ImportError as e:
    print(f"⚠️  RLFriendlyEnvironment not found, will use mock: {e}")

print()

# ── 2. Verify all curriculum levels have negotiation ──────────────────────
import train as _train_mod
curriculum = _train_mod.AdaptiveCurriculum()
print("📋 Curriculum Level → Negotiation Map:")
for lvl in range(5):
    curriculum.level = lvl
    cfg = curriculum.config_for_level()
    crisis_str = "+ crisis" if cfg['crisis_mode_enabled'] else "(stable)"
    neq_icon   = "✅" if cfg['negotiation_enabled'] else "❌"
    print(f"  Level {lvl}: negotiation={neq_icon}  crisis={cfg['crisis_mode_enabled']}  {crisis_str}")

curriculum.level = 0  # reset
print()

# ── 3. Verify challenge generator scenarios ───────────────────────────────
gen = _train_mod.ChallengeGenerator()
print("🎯 Challenge Generator Scenarios per Level:")
for lvl in range(5):
    plans = gen.get_plan_for_level(lvl, 2)
    names = [p['name'] for p in plans]
    print(f"  Level {lvl}: {names}")

print()
print("✅ All checks passed — negotiation active across ALL 5 levels")

## 4. Agent Inspection

Inspect Q-learning agent hyperparameters and negotiation capabilities.

In [ ]:
from src.rl_agent import RLDataLoaderAgent, RLDataCleanerAgent, RLMLTrainerAgent

agents = [
    RLDataLoaderAgent(),
    RLDataCleanerAgent(),
    RLMLTrainerAgent(),
]

print("🤖 Agent Hyperparameters")
print("─" * 60)
for ag in agents:
    rl = ag.rl if hasattr(ag, 'rl') else ag
    print(f"  {rl.name.upper()}")
    print(f"    learning_rate   : {rl.learning_rate}")
    print(f"    discount_factor : {rl.discount_factor}")
    print(f"    epsilon_start   : {rl.epsilon_start}")
    print(f"    epsilon_decay   : {rl.epsilon_decay}")
    print(f"    epsilon_min     : {rl.epsilon_min}")
    print(f"    replay_buf_max  : {rl.replay_buffer_max}")
    print()

print("📝 Negotiation Actions Available per Agent:")
sample_obs = {
    'hour': 3,
    'available_resources': {'gpu': {'available': 1, 'total': 1},
                             'cpu_cores': {'available': 4, 'total': 8},
                             'memory_gb': {'available': 16, 'total': 32}},
    'other_agents_status': {'data_loader': {'status': 'running'},
                             'data_cleaner': {'status': 'idle'},
                             'ml_trainer': {'status': 'waiting'}},
    'time_left_hours': 5,
    'my_tasks': {'pending': [{'task_id': 't1', 'priority': 'high'}], 'running': [], 'done': []},
}

for ag in agents:
    rl = ag.rl if hasattr(ag, 'rl') else ag
    action = rl.propose_action(sample_obs)
    print(f"  {rl.name}: {action}")

## 5. Main Training Run (50 Episodes · All 5 Levels · Negotiation ON)

Runs the full adaptive curriculum:
- **Level 0–1**: Negotiation-only (stable market + light outage)
- **Level 2–4**: Negotiation + crisis scenarios (urgent injections, compound failures)
- **Self-play duels**: Agents compete against their own past snapshots
- **Q-tables persist** across phases for continuous learning

In [ ]:
import importlib
import train as train_module

# Reload to pick up env-var changes made above
importlib.reload(train_module)

print("🚀 Starting CrisisCompute Training")
print(f"   Mode     : {train_module.TRAINING_AGENT_MODE.upper()}")
print(f"   Episodes : 50")
print(f"   Env      : {'REAL' if train_module.USE_REAL_ENV else 'MOCK'}")
print()

# Run training — results saved to ./results/
summary = train_module.train_agents(num_episodes=50)

## 6. Results & Learning Curves

In [ ]:
import json
import os

# Load all result files
def _load(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

training_results  = _load('results/training_results.json')
episode_metrics   = _load('results/episode_metrics.json')
selfplay_report   = _load('results/selfplay_report.json')
holdout_eval      = _load('results/holdout_evaluation.json')
baseline_cmp      = _load('results/baseline_comparison.json')
judge_summary     = _load('results/judge_summary.json')
theme4_summary    = _load('results/theme4_summary.json')

n_eps = len(training_results) if training_results else 0
rewards = [r['total_reward'] for r in training_results] if training_results else []

print(f"✅ Loaded {n_eps} episodes")
if rewards:
    from statistics import mean
    print(f"   First episode reward  : {rewards[0]:.1f}")
    print(f"   Last episode reward   : {rewards[-1]:.1f}")
    print(f"   Peak reward           : {max(rewards):.1f} (ep {rewards.index(max(rewards))+1})")
    print(f"   Mean reward           : {mean(rewards):.1f}")
    delta = rewards[-1] - rewards[0]
    pct   = delta / abs(rewards[0]) * 100 if rewards[0] != 0 else 0
    print(f"   Improvement (first→last): {delta:+.1f} ({pct:+.1f}%)")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

if not rewards:
    print("No rewards to plot")
else:
    episodes = list(range(1, len(rewards) + 1))

    # ── Rolling average ───────────────────────────────────────────────────
    window = 5
    rolling = []
    for i in range(len(rewards)):
        w_slice = rewards[max(0, i - window + 1): i + 1]
        rolling.append(sum(w_slice) / len(w_slice))

    # ── Curriculum level bands ────────────────────────────────────────────
    level_colors = ['#e8f5e9', '#fff9c4', '#ffe0b2', '#fce4ec', '#e8eaf6']
    level_labels = ['Level 0\nStable', 'Level 1\nLight Outage',
                    'Level 2\nUrgent', 'Level 3\nEarly Outage', 'Level 4\nCompound']

    curriculum_levels = None
    if training_results:
        curriculum_levels = [r.get('curriculum_level', 0) for r in training_results]

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # Top: reward curve
    ax = axes[0]
    ax.set_facecolor('#f9f9f9')

    # Shade by curriculum level
    if curriculum_levels:
        prev_lvl = curriculum_levels[0]
        seg_start = 0
        for i, lvl in enumerate(curriculum_levels + [None]):
            if lvl != prev_lvl or lvl is None:
                color = level_colors[min(prev_lvl, len(level_colors)-1)]
                ax.axvspan(seg_start + 1, i + 1, alpha=0.35, color=color, zorder=0)
                seg_start = i
                prev_lvl = lvl

    # League duel episodes
    duel_eps = [i+1 for i, r in enumerate(training_results or [])
                if r.get('league_duel', False)]
    for dep in duel_eps:
        ax.axvline(dep, color='#9c27b0', alpha=0.4, lw=1, linestyle='--')

    ax.plot(episodes, rewards, 'o-', color='#90caf9', lw=1.5, ms=3,
            alpha=0.7, label='Episode reward')
    ax.plot(episodes, rolling, '-', color='#1565c0', lw=2.5,
            label=f'{window}-ep rolling avg')

    # Trend line
    if len(rewards) >= 4:
        z = np.polyfit(episodes, rewards, 1)
        p = np.poly1d(z)
        ax.plot(episodes, p(episodes), '--', color='#e53935', lw=1.5,
                alpha=0.8, label=f'Trend (slope={z[0]:+.2f}/ep)')

    ax.set_xlabel('Episode', fontsize=12)
    ax.set_ylabel('Total Reward', fontsize=12)
    ax.set_title('CrisisCompute — Reward Curve (50 Episodes · Negotiation ON All Levels)', fontsize=13)

    # Legend patches for curriculum bands
    band_patches = [mpatches.Patch(color=level_colors[i], label=level_labels[i], alpha=0.6)
                    for i in range(len(level_labels))]
    if duel_eps:
        band_patches.append(mpatches.Patch(color='#9c27b0', alpha=0.4, label='League duel'))
    ax.legend(handles=band_patches + ax.get_lines()[:3], loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3)

    # Bottom: fairness + belief accuracy
    ax2 = axes[1]
    ax2.set_facecolor('#f9f9f9')
    if episode_metrics:
        fairness  = [m.get('fairness', 0.0) for m in episode_metrics]
        belief    = [m.get('belief_accuracy', 0.0) for m in episode_metrics]
        neq_health= [m.get('negotiation_health', 0.0) for m in episode_metrics]
        eps_m     = list(range(1, len(episode_metrics) + 1))
        ax2.plot(eps_m, fairness,   '-', color='#43a047', lw=2, label='Fairness score')
        ax2.plot(eps_m, belief,     '-', color='#fb8c00', lw=2, label='Belief accuracy')
        ax2.plot(eps_m, neq_health, '-', color='#8e24aa', lw=2, label='Negotiation health')
    ax2.set_xlabel('Episode', fontsize=12)
    ax2.set_ylabel('Score [0–1]', fontsize=12)
    ax2.set_title('Negotiation Quality Metrics Across Episodes', fontsize=13)
    ax2.set_ylim(0, 1.05)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('results/reward_curve_colab.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved: results/reward_curve_colab.png")

In [ ]:
# ── Curriculum Level Progression ─────────────────────────────────────────
if selfplay_report:
    lvl_summary = selfplay_report.get('level_progression_summary', {})
    ch_history  = selfplay_report.get('curriculum_history', [])

    print("📈 Curriculum Level Progression Summary")
    print("─" * 50)
    for lvl_key, stats in sorted(lvl_summary.items()):
        print(f"  {lvl_key}: episodes={stats['episodes']:3d}  mean_reward={stats['mean_reward']:.1f}")

    if ch_history:
        print()
        print("🗂️  Phase Promotion History")
        print("─" * 60)
        for entry in ch_history:
            promo = "⬆️ PROMOTED" if entry.get('promoted') else "  staying"
            print(
                f"  Phase {entry['phase']:2d} | Level {entry['level']} | "
                f"reward={entry.get('avg_total_reward', 0):.1f} | "
                f"fairness={entry.get('fairness', 0):.2f} | {promo}"
            )
else:
    print("ℹ️  No selfplay report (LLM-only mode or self-improvement disabled)")

## 7. Negotiation Analysis

In [ ]:
import matplotlib.pyplot as plt
from statistics import mean

if training_results:
    # Extract negotiation metrics per episode
    neg_data = []
    for r in training_results:
        neg_data.append({
            'ep':          r['episode'],
            'level':       r.get('curriculum_level', 0),
            'fairness':    r.get('avg_fairness_score', 0.0),
            'belief':      r.get('avg_belief_accuracy', 0.0),
            'contracts_k': r.get('contracts_kept', 0),
            'contracts_b': r.get('contracts_broken', 0),
            'coalitions':  r.get('coalitions_formed', 0),
            'conflicts':   r.get('conflict_count', 0),
            'renegotiate': r.get('renegotiation_count', 0),
            'urgency':     r.get('urgency_response_score', 1.0),
            'emergency':   r.get('emergency_charter_count', 0),
            'duel':        r.get('league_duel', False),
        })

    main_eps = [d for d in neg_data if not d['duel']]

    # ── Per-level fairness ────────────────────────────────────────────────
    level_fairness = {}
    for d in main_eps:
        lvl = d['level']
        level_fairness.setdefault(lvl, []).append(d['fairness'])

    print("⚖️  Mean Fairness Score by Curriculum Level (Negotiation ON all levels):")
    print("─" * 50)
    level_names = ['L0 Stable', 'L1 Light Outage', 'L2 Urgent', 'L3 Early Outage', 'L4 Compound']
    for lvl in sorted(level_fairness.keys()):
        vals = level_fairness[lvl]
        name = level_names[lvl] if lvl < len(level_names) else f'L{lvl}'
        print(f"  {name:20s}: {mean(vals):.3f}  (n={len(vals)} episodes)")

    print()

    # ── Aggregate negotiation stats ──────────────────────────────────────
    total_kept    = sum(d['contracts_k'] for d in main_eps)
    total_broken  = sum(d['contracts_b'] for d in main_eps)
    total_coal    = sum(d['coalitions']  for d in main_eps)
    total_conf    = sum(d['conflicts']   for d in main_eps)
    total_renegot = sum(d['renegotiate'] for d in main_eps)
    total_emerg   = sum(d['emergency']   for d in main_eps)
    contract_rate = total_kept / max(1, total_kept + total_broken)

    print("📊 Negotiation Aggregate Statistics (main episodes only):")
    print(f"   Contracts kept       : {total_kept}")
    print(f"   Contracts broken     : {total_broken}")
    print(f"   Contract keep rate   : {contract_rate:.1%}")
    print(f"   Coalitions formed    : {total_coal}")
    print(f"   Conflicts            : {total_conf}")
    print(f"   Renegotiations       : {total_renegot}")
    print(f"   Emergency charters   : {total_emerg}")

    # ── Action distribution ───────────────────────────────────────────────
    print()
    if summary and isinstance(summary, dict):
        hist = summary.get('action_distribution', {})
        if hist:
            total_acts = sum(hist.values())
            print("🎯 Action Distribution (all agents combined):")
            for act, cnt in sorted(hist.items(), key=lambda kv: -kv[1]):
                bar = '█' * int(cnt / total_acts * 40)
                print(f"   {act:20s}: {cnt:5d}  ({cnt/total_acts*100:5.1f}%)  {bar}")
else:
    print("No training results loaded yet — run the training cell first.")

## 8. Holdout Evaluation: Trained vs Untrained

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if holdout_eval:
    trained = holdout_eval.get('trained_summary', {})
    fresh   = holdout_eval.get('fresh_summary', {})
    delta   = holdout_eval.get('delta', {})

    metrics = [
        ('avg_total_reward',           'Total Reward'),
        ('avg_completion_rate',        'Completion Rate'),
        ('avg_on_time_rate',           'On-Time Rate'),
        ('avg_fairness_score',         'Fairness Score'),
        ('avg_belief_accuracy',        'Belief Accuracy'),
        ('avg_urgency_response_score', 'Urgency Response'),
    ]

    print("🎯 Holdout Evaluation — Compound Crisis Scenarios")
    print(f"   Scenario: {[p['name'] for p in holdout_eval.get('holdout_plan', [])]}")
    print()
    print(f"  {'Metric':<28} {'Trained':>10} {'Untrained':>10} {'Delta':>10}")
    print("  " + "─" * 60)
    for key, label in metrics:
        t_val = trained.get(key, 0.0)
        f_val = fresh.get(key, 0.0)
        d_val = delta.get(key, 0.0)
        sign  = "✅" if d_val > 0 else ("⚠️ " if d_val < 0 else "─")
        print(f"  {label:<28} {t_val:>10.3f} {f_val:>10.3f} {d_val:>+10.3f} {sign}")

    # Bar chart
    fig, ax = plt.subplots(figsize=(12, 5))
    labels  = [m[1] for m in metrics]
    t_vals  = [trained.get(m[0], 0.0) for m in metrics]
    f_vals  = [fresh.get(m[0], 0.0)   for m in metrics]

    x = np.arange(len(labels))
    w = 0.35
    bars1 = ax.bar(x - w/2, t_vals, w, label='Trained (50 eps)', color='#1565c0', alpha=0.85)
    bars2 = ax.bar(x + w/2, f_vals, w, label='Untrained baseline', color='#ef9a9a', alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=10)
    ax.set_ylabel('Score')
    ax.set_title('Holdout Evaluation: Trained Agent vs Untrained Baseline\n(Compound Crisis — Negotiation Enabled)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    for bar in bars1:
        h = bar.get_height()
        ax.annotate(f'{h:.2f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
    for bar in bars2:
        h = bar.get_height()
        ax.annotate(f'{h:.2f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

    plt.tight_layout()
    plt.savefig('results/holdout_comparison_colab.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved: results/holdout_comparison_colab.png")
else:
    print("Holdout evaluation not available (run RL/Hybrid mode with SELF_IMPROVEMENT_ENABLED=true)")

## 9. Baseline Comparison: Negotiation ON vs OFF

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if baseline_cmp:
    neq_run  = baseline_cmp.get('negotiation_enabled_run', {})
    base_run = baseline_cmp.get('baseline_run_negotiation_disabled', {})
    delta    = baseline_cmp.get('delta', {})

    metrics = [
        ('avg_total_reward',    'Total Reward'),
        ('avg_completion_rate', 'Completion Rate'),
        ('avg_on_time_rate',    'On-Time Rate'),
        ('avg_fairness_score',  'Fairness Score'),
        ('avg_belief_accuracy', 'Belief Accuracy'),
    ]

    print("⚡ Negotiation ON vs OFF — Impact Analysis")
    print()
    print(f"  {'Metric':<28} {'Neg ON':>10} {'Neg OFF':>10} {'Delta':>10}")
    print("  " + "─" * 60)
    for key, label in metrics:
        n_val = neq_run.get(key, 0.0)
        b_val = base_run.get(key, 0.0)
        d_val = delta.get(key, 0.0)
        sign  = "✅" if d_val > 0 else ("⚠️ " if d_val < 0 else "─")
        print(f"  {label:<28} {n_val:>10.3f} {b_val:>10.3f} {d_val:>+10.3f} {sign}")

    fig, ax = plt.subplots(figsize=(11, 4))
    labels = [m[1] for m in metrics]
    n_vals = [neq_run.get(m[0], 0.0)  for m in metrics]
    b_vals = [base_run.get(m[0], 0.0) for m in metrics]
    x = np.arange(len(labels))
    w = 0.35
    ax.bar(x - w/2, n_vals, w, label='Negotiation ON',  color='#43a047', alpha=0.85)
    ax.bar(x + w/2, b_vals, w, label='Negotiation OFF', color='#e53935', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=10)
    ax.set_ylabel('Score')
    ax.set_title('Negotiation Enabled vs Disabled — Value of Negotiation Protocol')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/baseline_comparison_colab.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved: results/baseline_comparison_colab.png")
else:
    print("Baseline comparison not available yet — run training first")

## 10. Q-Table Analysis

In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np

q_table_dir = 'q_tables'
agents_to_check = ['data_loader', 'data_cleaner', 'ml_trainer']

print("💾 Q-Table Analysis")
print("─" * 60)

q_stats = {}
for agent_name in agents_to_check:
    qt_path = os.path.join(q_table_dir, f'{agent_name}_q_table.json')
    ep_path = os.path.join(q_table_dir, f'{agent_name}_epsilon.json')

    if not os.path.exists(qt_path):
        print(f"  {agent_name}: Q-table not found (RL/Hybrid mode required)")
        continue

    with open(qt_path) as f:
        q_table = json.load(f)

    epsilon = None
    if os.path.exists(ep_path):
        with open(ep_path) as f:
            epsilon = json.load(f).get('epsilon')

    all_q_vals = [v for state_actions in q_table.values()
                  for v in state_actions.values()]

    # Action frequency: count which action has the highest Q for each state
    best_actions = {}
    for state, actions in q_table.items():
        if actions:
            best_act = max(actions.items(), key=lambda kv: kv[1])[0]
            best_actions[best_act] = best_actions.get(best_act, 0) + 1

    stats = {
        'states': len(q_table),
        'q_mean': np.mean(all_q_vals) if all_q_vals else 0.0,
        'q_max':  np.max(all_q_vals)  if all_q_vals else 0.0,
        'q_min':  np.min(all_q_vals)  if all_q_vals else 0.0,
        'q_std':  np.std(all_q_vals)  if all_q_vals else 0.0,
        'epsilon': epsilon,
        'best_actions': best_actions,
    }
    q_stats[agent_name] = stats

    print(f"  {agent_name.upper()}")
    print(f"    States explored : {stats['states']}")
    print(f"    Q-value range   : [{stats['q_min']:.2f}, {stats['q_max']:.2f}]")
    print(f"    Q-value mean    : {stats['q_mean']:.2f}  std={stats['q_std']:.2f}")
    if epsilon is not None:
        print(f"    Epsilon (final) : {epsilon:.4f}")
    print(f"    Preferred actions (by best-Q count):")
    total_states = sum(best_actions.values())
    for act, cnt in sorted(best_actions.items(), key=lambda kv: -kv[1]):
        bar = '▓' * int(cnt / max(1, total_states) * 30)
        print(f"      {act:20s}: {cnt:4d} ({cnt/max(1,total_states)*100:5.1f}%) {bar}")
    print()

if not q_stats:
    print("  No Q-tables found — ensure TRAINING_AGENT_MODE=rl or hybrid and run training")

## 11. HuggingFace TRL — GRPO Training Loop

This section demonstrates how CrisisCompute integrates with **HuggingFace TRL** for LLM post-training via **GRPO** (Group Relative Policy Optimization). The RL environment provides the reward signal; TRL handles gradient updates on the LLM.

In [ ]:
# Install TRL + Unsloth (Colab only)
import subprocess, sys

def _try_install(pkg):
    try:
        __import__(pkg.split('>=')[0].split('[')[0].replace('-', '_'))
        return True
    except ImportError:
        return False

trl_available      = _try_install('trl')
unsloth_available  = _try_install('unsloth')
transformers_avail = _try_install('transformers')

print(f"TRL available        : {trl_available}")
print(f"Unsloth available    : {unsloth_available}")
print(f"Transformers available: {transformers_avail}")

if not trl_available:
    print("\nInstalling TRL and dependencies...")
    !pip install -q trl transformers accelerate peft bitsandbytes
    print("✅ TRL installed")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CrisisCompute Reward Function for TRL / GRPO
#
# The environment step() returns rewards[i] per agent. We aggregate them
# into a scalar reward for the LLM's generation and normalize to [-1, 1].
# ─────────────────────────────────────────────────────────────────────────

import json
import sys
import os
sys.path.insert(0, os.getcwd())

# Reward normalisation constants (from empirical episode range)
REWARD_MIN = 600.0
REWARD_MAX = 800.0

def normalize_reward(raw_reward: float) -> float:
    """Map raw episode reward [600, 800] → [-1, 1]."""
    clipped = max(REWARD_MIN, min(REWARD_MAX, raw_reward))
    return 2.0 * (clipped - REWARD_MIN) / (REWARD_MAX - REWARD_MIN) - 1.0


def crisis_compute_reward_fn(completions, prompts=None, **kwargs):
    """
    TRL-compatible reward function for CrisisCompute.

    Each `completion` is the full text output from the LLM agent for a
    single episode step. We:
      1. Try to parse the completion as a JSON action dict.
      2. Run a 1-step environment rollout with that action.
      3. Return the normalised reward as the GRPO scalar signal.

    Falls back to a shaping heuristic if the action cannot be parsed:
      - Valid JSON with expected keys  → 0.0
      - Invalid JSON                   → -0.5 (format penalty)
    """
    rewards_out = []

    try:
        from satya_env.rl_environment import RLFriendlyEnvironment
        env = RLFriendlyEnvironment(config_dir=None, seed=42)
        env.reset()
    except Exception:
        env = None

    for completion in completions:
        text = completion if isinstance(completion, str) else (
            completion[0].get('content', '') if isinstance(completion, list) else str(completion)
        )

        # ── 1. Try to parse the action ────────────────────────────────────
        action = None
        try:
            # LLM may wrap JSON in markdown ```json … ```
            text_clean = text.strip()
            if text_clean.startswith('```'):
                text_clean = text_clean.split('```')[1]
                if text_clean.startswith('json'):
                    text_clean = text_clean[4:]
            action = json.loads(text_clean)
        except (json.JSONDecodeError, IndexError):
            # Can't parse → format penalty
            rewards_out.append(-0.5)
            continue

        # ── 2. Validate action keys ───────────────────────────────────────
        required_keys = {'action'}
        if not required_keys.issubset(action.keys()):
            rewards_out.append(-0.25)
            continue

        # ── 3. Run environment step ───────────────────────────────────────
        if env is not None:
            try:
                actions_dict = {
                    'data_loader':  action,
                    'data_cleaner': action,
                    'ml_trainer':   action,
                }
                _, step_rewards, _, _ = env.step(actions_dict)
                raw_r = sum(step_rewards)
                rewards_out.append(normalize_reward(raw_r))
            except Exception:
                rewards_out.append(0.0)
        else:
            # No env available — use heuristic shaping
            act_type = action.get('action', '')
            if act_type == 'run_task':
                cores = int(action.get('cores_needed', 4))
                # Prefer minimal resource usage (negotiation-aligned)
                shape = 0.3 if cores <= 2 else (0.1 if cores <= 4 else -0.2)
            elif act_type == 'wait':
                shape = 0.0
            elif act_type == 'negotiate':
                shape = 0.4  # Bonus for explicit negotiation
            else:
                shape = -0.1
            rewards_out.append(shape)

    return rewards_out


# Quick sanity check
test_completions = [
    '{"action": "run_task", "task_id": "t1", "cores_needed": 2, "gpu_needed": 0}',  # good
    '{"action": "negotiate", "proposal": "share_gpu"}',                              # bonus
    'This is not JSON at all',                                                        # bad
    '{"action": "wait"}',                                                             # neutral
]
test_rewards = crisis_compute_reward_fn(test_completions)
print("Reward function sanity check:")
for comp, rew in zip(test_completions, test_rewards):
    label = comp[:50].replace('\n', ' ')
    print(f"  reward={rew:+.3f}  | {label}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# TRL GRPO Training Loop (runs when TRL is available)
#
# Uses a small instruction-tuned model and trains it to produce better
# negotiation actions via the CrisisCompute reward function above.
# ─────────────────────────────────────────────────────────────────────────

try:
    import torch
    from trl import GRPOConfig, GRPOTrainer
    from transformers import AutoTokenizer, AutoModelForCausalLM
    TRL_OK = True
except ImportError:
    TRL_OK = False
    print("TRL / transformers not available — skipping GRPO training")
    print("Install with: pip install trl transformers accelerate peft bitsandbytes")

if TRL_OK:
    import os

    MODEL_NAME  = os.getenv('GRPO_MODEL', 'Qwen/Qwen2.5-0.5B-Instruct')  # tiny model for demo
    OUTPUT_DIR  = 'results/trl_grpo_output'
    NUM_GRPO_EP = int(os.getenv('GRPO_EPISODES', '2'))   # set higher for real training

    print(f"Loading model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model     = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto',
    )
    print(f"✅ Model loaded on {'GPU' if torch.cuda.is_available() else 'CPU'}")

    # ── Build prompt dataset ──────────────────────────────────────────────
    # Each prompt describes the current negotiation state and asks the
    # agent to output a JSON action.
    SYSTEM_PROMPT = (
        "You are a compute resource negotiation agent in CrisisCompute. "
        "You manage tasks for a data pipeline under constrained GPU/CPU resources. "
        "Output ONLY a valid JSON action object with keys: "
        "action (run_task|wait|negotiate), task_id, cores_needed, gpu_needed, memory_needed."
    )

    scenario_prompts = [
        "Hour 1 of 8. Available: CPU=6, GPU=1, Memory=28GB. "
        "Pending tasks: [{task_id: 'load_batch_1', priority: 'high', cores_min: 2, gpu_needed: 0}]. "
        "Other agents: data_cleaner=idle, ml_trainer=waiting. No crisis. "
        "What action do you take?",

        "Hour 3 of 8. GPU OUTAGE at hour 3. Available: CPU=4, GPU=0, Memory=20GB. "
        "Pending tasks: [{task_id: 'train_model', priority: 'critical', gpu_needed: 1}]. "
        "ml_trainer wants the GPU. data_loader has CPU. "
        "Negotiate or wait for GPU to come back online. What action do you take?",

        "Hour 5 of 8. URGENT task injected: incident_model_rebuild (deadline: 2h). "
        "Available: CPU=3, GPU=1, Memory=12GB. "
        "Other agents: data_loader=done, data_cleaner=running (needs 4 CPU). "
        "Form a coalition or prioritize the urgent task. What action do you take?",

        "Hour 7 of 8. Final hour. CPU=2, GPU=1, Memory=8GB. "
        "3 tasks still pending. Fairness score = 0.62 (below target 0.78). "
        "Renegotiate allocation or request resource release from finished agents.",
    ]

    def make_prompt(scenario_text: str) -> str:
        return (
            f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
            f"<|user|>\n{scenario_text}<|end|>\n"
            f"<|assistant|>\n"
        )

    from datasets import Dataset
    train_data = Dataset.from_dict({
        'prompt': [make_prompt(p) for p in scenario_prompts * 4],  # 16 examples
    })

    # ── GRPO Config ───────────────────────────────────────────────────────
    grpo_config = GRPOConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=NUM_GRPO_EP,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=5e-6,
        max_new_tokens=128,
        temperature=0.7,
        logging_steps=1,
        save_steps=50,
        report_to='none',
        num_generations=4,   # GRPO group size
    )

    # ── Trainer ───────────────────────────────────────────────────────────
    trainer = GRPOTrainer(
        model=model,
        args=grpo_config,
        reward_funcs=crisis_compute_reward_fn,
        train_dataset=train_data,
        tokenizer=tokenizer,
    )

    print(f"\n🎓 Starting GRPO training ({NUM_GRPO_EP} epochs on {len(train_data)} prompts)")
    trainer.train()
    print("✅ GRPO training complete")
    trainer.save_model(OUTPUT_DIR)
    print(f"✅ Model saved to {OUTPUT_DIR}")

## 12. Unsloth Fast-Training (Optional)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Unsloth-accelerated GRPO training
# Uses 4-bit QLoRA + Unsloth optimisations for 2-4× faster training.
# ─────────────────────────────────────────────────────────────────────────

try:
    from unsloth import FastLanguageModel, PatchFastRL
    PatchFastRL('GRPO')
    UNSLOTH_OK = True
    print("✅ Unsloth available — using fast 4-bit training")
except ImportError:
    UNSLOTH_OK = False
    print("Unsloth not installed — skipping (install with: pip install unsloth)")

if UNSLOTH_OK:
    import os
    UNSLOTH_MODEL = os.getenv('UNSLOTH_MODEL', 'unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit')

    model_us, tokenizer_us = FastLanguageModel.from_pretrained(
        model_name=UNSLOTH_MODEL,
        max_seq_length=512,
        load_in_4bit=True,
        fast_inference=True,
    )
    model_us = FastLanguageModel.get_peft_model(
        model_us,
        r=16,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                         'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=16,
        use_gradient_checkpointing='unsloth',
    )

    from trl import GRPOConfig, GRPOTrainer
    from datasets import Dataset

    # Re-use prompts from Section 11
    unsloth_data = Dataset.from_dict({'prompt': [make_prompt(p) for p in scenario_prompts * 4]})

    grpo_cfg_us = GRPOConfig(
        output_dir='results/unsloth_grpo_output',
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=5e-6,
        max_new_tokens=128,
        temperature=0.7,
        logging_steps=1,
        report_to='none',
        num_generations=4,
    )

    trainer_us = GRPOTrainer(
        model=model_us,
        args=grpo_cfg_us,
        reward_funcs=crisis_compute_reward_fn,
        train_dataset=unsloth_data,
        tokenizer=tokenizer_us,
    )

    print("🚀 Unsloth GRPO training starting...")
    trainer_us.train()
    print("✅ Unsloth training complete")

## 13. Final Summary & Export

In [ ]:
import json, os
from statistics import mean, pstdev

print("═" * 70)
print("  CrisisCompute — Training Complete")
print("═" * 70)

if judge_summary:
    cfg = judge_summary.get('config', {})
    agg = judge_summary.get('aggregate', {})
    print()
    print("📋 Judge Summary:")
    print(f"   Seeds                  : {cfg.get('seeds')}")
    print(f"   Negotiation enabled    : {cfg.get('negotiation_enabled')}")
    print(f"   Episodes               : {cfg.get('episodes_per_seed')}")
    print(f"   Reward mean ± std      : {agg.get('reward_mean', 0):.2f} ± {agg.get('reward_std', 0):.2f}")
    print(f"   Completion mean ± std  : {agg.get('completion_mean', 0):.3f} ± {agg.get('completion_std', 0):.3f}")
    print(f"   On-time mean ± std     : {agg.get('on_time_mean', 0):.3f} ± {agg.get('on_time_std', 0):.3f}")

if theme4_summary:
    lvl_prog = theme4_summary.get('level_progression_summary', {})
    print()
    print("🎯 Self-Improvement (Theme #4) Level Progression:")
    for lvl_key, stats in sorted(lvl_prog.items()):
        print(f"   {lvl_key}: eps={stats['episodes']:3d}  mean_reward={stats['mean_reward']:.1f}")

    held_delta = theme4_summary.get('holdout_delta', {})
    if held_delta:
        print()
        print("🧪 Holdout Delta (trained − untrained):")
        for k, v in held_delta.items():
            sign = "✅" if v > 0 else ("⚠️ " if v < 0 else "─")
            print(f"   {k:<35}: {v:+.4f} {sign}")

print()
print("📁 Saved artefacts:")
result_files = [
    'results/reward_curve.png',
    'results/metrics_dashboard.png',
    'results/reward_curve_colab.png',
    'results/holdout_comparison_colab.png',
    'results/baseline_comparison_colab.png',
    'results/training_results.json',
    'results/episode_metrics.json',
    'results/selfplay_report.json',
    'results/holdout_evaluation.json',
    'results/baseline_comparison.json',
    'results/judge_summary.json',
    'results/theme4_summary.json',
]
for f in result_files:
    exists = os.path.exists(f)
    icon   = '✅' if exists else '❌'
    size   = f"{os.path.getsize(f)/1024:.1f} KB" if exists else 'missing'
    print(f"   {icon} {f}  ({size})")

print()
print("═" * 70)
print("  🏁 Notebook complete! Upload results/ to your HF Space.")
print("═" * 70)

In [ ]:
# Download results as a zip (Colab only)
try:
    from google.colab import files
    import zipfile, os

    zip_path = 'crisiscompute_results.zip'
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for root, dirs, fnames in os.walk('results'):
            for fname in fnames:
                fpath = os.path.join(root, fname)
                zf.write(fpath)
        for root, dirs, fnames in os.walk('q_tables'):
            for fname in fnames:
                fpath = os.path.join(root, fname)
                zf.write(fpath)

    files.download(zip_path)
    print(f"✅ Downloading {zip_path}")
except Exception as e:
    print(f"Download skipped (not in Colab or no results yet): {e}")